<a href="https://colab.research.google.com/github/lohaniSatwik/steam-games-data-mining/blob/master/Code/section8_feature_engineering_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 8 — Feature Engineering Experiment
**IE500 Data Mining | Team 9 – Brewed Clusters**

> **Google Colab notebook.** Run all cells top to bottom.

## Purpose
The Section 7 error analysis identified that **Mixed → Good** (42.4% of Mixed) is the dominant error.
The top separating features between Good and Mixed were: `tag_2D`, `era_2020+`, `cat_Steam Cloud`,
`tag_Singleplayer`, `cat_Full controller support`.

This notebook tests whether **8 new derived features** built from those signals improve Random Forest performance.

## New features (147 → 155)

| Feature | Type | Rationale |
|---------|------|-----------|
| `n_tags` | count (scaled) | Number of tags — more tags = more discoverable, tends toward Good |
| `n_categories` | count (scaled) | Number of Steam feature categories — proxy for release polish |
| `n_genres` | count (scaled) | Number of genres listed |
| `polish_index` | sum (scaled) | Steam Cloud + Full controller support + Steam Achievements + Singleplayer + Mac — quality signal count |
| `recent_2d` | binary | `tag_2D × era_2020+` — recent 2D games are disproportionately Good (diff=+0.172×+0.159) |
| `price_x_recent` | interaction (scaled) | `log_price × era_2020+` — pricey recent games are almost always Good |
| `solo_only` | binary | `tag_Singleplayer × (1 − cat_Multi-player)` — focused singleplayer = clearer audience = cleaner reception |
| `platform_breadth` | count (scaled) | Windows + Mac + Linux — multi-platform = more developer commitment |

## Baseline to beat
- **Original RF (147 features):** Macro F1 = **0.5100**
- Same nested CV setup: 5-fold outer, 3-fold inner GridSearchCV
- Same param grid, same random seed — only the dataset changes

In [ ]:
import os
if not os.path.exists('steam-games-data-mining'):
    !git clone https://github.com/lohaniSatwik/steam-games-data-mining.git
else:
    !git -C steam-games-data-mining pull
DATA_DIR = 'steam-games-data-mining/Datasets'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE   = 42
CLASS_ORDER    = ['Good', 'Mixed', 'Bad']
CLASS_COLORS   = {'Good': 'steelblue', 'Mixed': 'sandybrown', 'Bad': 'salmon'}
BASELINE_RF    = 0.5100   # original RF with 147 features
BASELINE_LR    = 0.4355
BASELINE_SVM   = 0.4693
NEW_FEATURES   = ['n_tags', 'n_categories', 'n_genres', 'polish_index',
                  'recent_2d', 'price_x_recent', 'solo_only', 'platform_breadth']
print('Libraries loaded.')

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train_multiclass_fe.csv')
test  = pd.read_csv(f'{DATA_DIR}/test_multiclass_fe.csv')

X_train = train.drop(columns=['label_multiclass'])
y_train = train['label_multiclass']
X_test  = test.drop(columns=['label_multiclass'])
y_test  = test['label_multiclass']

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Features: {X_train.shape[1]}  (original: 147, new: {X_train.shape[1]-147})')
print()
print('New features present:', all(f in X_train.columns for f in NEW_FEATURES))
print()
print('New feature statistics (train):')
print(X_train[NEW_FEATURES].describe().round(3).to_string())
print()
print('Class distribution (train):')
vc = y_train.value_counts()
for cls in CLASS_ORDER:
    print(f'  {cls:6s}: {vc[cls]:6,d}  ({vc[cls]/len(y_train)*100:.1f}%)')

## Nested Cross-Validation
Identical setup to Section 4b — only the feature set changes.

Expected runtime on Colab: **~20–40 minutes**

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    'n_estimators':     [100, 200],
    'max_depth':        [None, 10, 20],
    'min_samples_leaf': [1, 4]
}

outer_scores    = []
best_params_log = []

print('Running 5-fold nested CV (inner 3-fold GridSearchCV)...\n')

for fold, (tr_idx, val_idx) in tqdm(
        enumerate(outer_cv.split(X_train, y_train), 1),
        total=5, desc='Outer folds'):

    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    gs = GridSearchCV(
        RandomForestClassifier(
            class_weight='balanced', max_features='sqrt',
            random_state=RANDOM_STATE, n_jobs=-1
        ),
        param_grid, cv=inner_cv, scoring='f1_macro', n_jobs=-1, refit=True
    )
    gs.fit(X_tr, y_tr)

    y_pred = gs.predict(X_val)
    f1     = f1_score(y_val, y_pred, average='macro')
    outer_scores.append(f1)
    best_params_log.append(gs.best_params_)
    print(f'  Fold {fold} | Macro F1: {f1:.4f} | {gs.best_params_}')

print(f'\nNested CV (FE, 155 features) →  Macro F1: {np.mean(outer_scores):.4f} ± {np.std(outer_scores):.4f}')
print(f'Original RF  (147 features)  →  Macro F1: {BASELINE_RF:.4f}')
print(f'Improvement                  →  {np.mean(outer_scores) - BASELINE_RF:+.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
folds = list(range(1, 6))
ax.bar(folds, outer_scores, color='mediumseagreen', edgecolor='white', alpha=0.85)
ax.axhline(np.mean(outer_scores), color='darkgreen', linestyle='--', linewidth=1.5,
           label=f'RF+FE Mean = {np.mean(outer_scores):.4f}')
ax.axhline(BASELINE_RF, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Original RF = {BASELINE_RF:.4f}')
ax.axhline(BASELINE_SVM, color='darkorange', linestyle='--', linewidth=1.5,
           label=f'SVM Baseline = {BASELINE_SVM:.4f}')
ax.axhline(np.mean(outer_scores) + np.std(outer_scores), color='darkgreen',
           linestyle=':', linewidth=1, alpha=0.6)
ax.axhline(np.mean(outer_scores) - np.std(outer_scores), color='darkgreen',
           linestyle=':', linewidth=1, alpha=0.6,
           label=f'+/-1 std = {np.std(outer_scores):.4f}')
ax.set_xlabel('Outer Fold')
ax.set_ylabel('Macro F1')
ax.set_title('RF + Feature Engineering — Nested CV Macro F1 per Fold')
ax.set_xticks(folds)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('Best hyperparameters across outer folds:')
param_counts = Counter([str(p) for p in best_params_log])
for params, count in param_counts.most_common():
    print(f'  {count:2d} fold(s): {params}')

## Final Model
Re-run `GridSearchCV` on the **full training set**, then evaluate on the held-out test set **once**.

In [ ]:
print('Fitting final model on full training set...\n')

final_gs = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced', max_features='sqrt',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    param_grid, cv=inner_cv, scoring='f1_macro', n_jobs=-1, refit=True
)
final_gs.fit(X_train, y_train)

print(f'Best params      : {final_gs.best_params_}')
print(f'Best inner CV F1 : {final_gs.best_score_:.4f}')

## Test Set Evaluation
> Evaluate on `test_multiclass_fe.csv` **once only** — this is the final performance number.

In [ ]:
final_model = final_gs.best_estimator_
y_pred_test = final_model.predict(X_test)

test_macro_f1 = f1_score(y_test, y_pred_test, average='macro')
print(f'Test Macro F1 (RF + FE, 155 features) : {test_macro_f1:.4f}')
print(f'Original RF   (147 features)           : {BASELINE_RF:.4f}')
print(f'Improvement                            : {test_macro_f1 - BASELINE_RF:+.4f}')
print(f'LR Baseline                            : {BASELINE_LR:.4f}')
print()
print('Classification Report (Test Set):')
print(classification_report(y_test, y_pred_test, labels=CLASS_ORDER, target_names=CLASS_ORDER))

In [ ]:
# ── Side-by-side: original RF vs RF+FE confusion matrices ────────────────────
# Refit original RF for comparison (same params, 147 features)
from sklearn.preprocessing import LabelEncoder
train_orig = pd.read_csv(f'{DATA_DIR}/train_multiclass.csv')
test_orig  = pd.read_csv(f'{DATA_DIR}/test_multiclass.csv')
X_train_orig = train_orig.drop(columns=['label_multiclass'])
y_train_orig = train_orig['label_multiclass']
X_test_orig  = test_orig.drop(columns=['label_multiclass'])
y_test_orig  = test_orig['label_multiclass']

rf_orig = RandomForestClassifier(
    n_estimators=200, max_depth=None, min_samples_leaf=4,
    max_features='sqrt', class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_orig.fit(X_train_orig, y_train_orig)
y_pred_orig = rf_orig.predict(X_test_orig)

cm_orig = confusion_matrix(y_test_orig, y_pred_orig, labels=CLASS_ORDER)
cm_fe   = confusion_matrix(y_test,      y_pred_test, labels=CLASS_ORDER)
cm_orig_norm = cm_orig.astype(float) / cm_orig.sum(axis=1, keepdims=True)
cm_fe_norm   = cm_fe.astype(float)   / cm_fe.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm_orig_norm, display_labels=CLASS_ORDER).plot(
    ax=axes[0], colorbar=False, cmap='Blues', values_format='.2f')
axes[0].set_title(f'Original RF (147 features)\nMacro F1 = {BASELINE_RF:.4f}', fontsize=11)

ConfusionMatrixDisplay(cm_fe_norm, display_labels=CLASS_ORDER).plot(
    ax=axes[1], colorbar=False, cmap='Greens', values_format='.2f')
axes[1].set_title(f'RF + Feature Engineering (155 features)\nMacro F1 = {test_macro_f1:.4f}', fontsize=11)

plt.suptitle('Row-Normalised Confusion Matrix: Original RF vs RF+FE\n'
             '(diagonal = recall per class — higher is better)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('Per-class recall comparison:')
print(f"{'Class':<8} {'Original RF':>14} {'RF + FE':>14} {'Change':>10}")
print('-' * 48)
for i, cls in enumerate(CLASS_ORDER):
    r_orig = cm_orig_norm[i, i]
    r_fe   = cm_fe_norm[i, i]
    print(f'{cls:<8} {r_orig:>14.3f} {r_fe:>14.3f} {r_fe-r_orig:>+10.3f}')

## Feature Importance — Did the New Features Help?
Gini importance shows where each feature contributed to splits.
New features are highlighted in green.

In [ ]:
feature_names = X_train.columns.tolist()
importances   = final_model.feature_importances_
imp_series    = pd.Series(importances, index=feature_names).sort_values(ascending=False)

top25 = imp_series.head(25)
colors = ['mediumseagreen' if f in NEW_FEATURES else 'steelblue' for f in top25.index]

fig, ax = plt.subplots(figsize=(10, 8))
top25.sort_values().plot(kind='barh', ax=ax, color=
    ['mediumseagreen' if f in NEW_FEATURES else 'steelblue' for f in top25.sort_values().index],
    alpha=0.85)
ax.set_title('RF+FE — Top 25 Feature Importances (Gini)\n'
             'Green = new engineered features | Blue = original features', fontsize=12)
ax.set_xlabel('Mean Decrease in Impurity')

# Legend
new_patch  = mpatches.Patch(color='mediumseagreen', label='New engineered features')
orig_patch = mpatches.Patch(color='steelblue',      label='Original features')
ax.legend(handles=[new_patch, orig_patch])

plt.tight_layout()
plt.show()

print('New features in Top 25:')
for feat in top25.index:
    if feat in NEW_FEATURES:
        rank = list(imp_series.index).index(feat) + 1
        print(f'  Rank {rank:2d}: {feat:25s}  importance={imp_series[feat]:.4f}')

print('\nAll new features and their importance rank:')
all_ranks = list(imp_series.index)
for feat in NEW_FEATURES:
    rank = all_ranks.index(feat) + 1
    print(f'  Rank {rank:3d}/{len(all_ranks)}: {feat:25s}  importance={imp_series[feat]:.4f}')

## Results Summary

| Metric | RF + FE (155 features) | Original RF (147 features) | Change |
|--------|:---:|:---:|:---:|
| Nested CV Macro F1 | — | **0.5093 ± 0.0073** | — |
| Test Macro F1 | — | **0.5100** | — |
| Good F1 | — | 0.77 | — |
| Mixed F1 | — | 0.43 | — |
| Bad F1 | — | 0.33 | — |

**New features added:**
- `n_tags`, `n_categories`, `n_genres` — count signals
- `polish_index` — quality signal composite
- `recent_2d` — interaction term (tag_2D × era_2020+)
- `price_x_recent` — interaction term (log_price × era_2020+)
- `solo_only` — focused singleplayer indicator
- `platform_breadth` — multi-platform commitment signal

**Interpretation:**

> Fill in after running.